# 04 · Producción agrícola: silver y gold
**Responsable:** P2 · **Código:** `src/silver_agricola.py`, `src/gold_agricola.py` · **Salidas:** `silver/agricola.parquet`, `gold/cultivo_municipio.parquet`, figura por cultivo

**Qué hace (silver)**
- Renombra a snake_case, convierte números (acepta `1234.5` y `1.234,5`) y cuenta los valores que no se pudieron leer, antes de convertirlos.
- Reglas que **rechazan:** departamento distinto de 15, código inválido, periodo mal escrito, producción o área faltantes (faltante ≠ cero), valores negativos y filas repetidas por municipio-cultivo-periodo-estado.
- Reglas que **solo alertan:** área cosechada > sembrada, producción sin área cosechada y rendimientos extremos. Se marcan en columnas `alerta_*` y no se borran.

**Qué hace (gold)**
- Suma los periodos A y B por municipio y cultivo. El rendimiento es **Σ producción / Σ área cosechada**, nunca el promedio de rendimientos. Si el área es 0, el rendimiento queda null.
- Concilia: la producción total de gold debe ser igual a la de silver.

**Advertencia:** sumar toneladas de cultivos distintos (papa fresca, café pergamino, caña) no da un valor económico.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [ ]:
from src.silver_agricola import construir_agricola
agr = construir_agricola()
agr.group_by("periodo").len()

In [ ]:
from src.gold_agricola import construir_cultivo_municipio, resumen_departamental_por_cultivo
cm = construir_cultivo_municipio(agr)
resumen = resumen_departamental_por_cultivo(cm)
resumen.head(15)

In [ ]:
from src import graficos
graficos.produccion_rendimiento_por_cultivo(resumen.to_pandas(), ANIO_EVA)

In [ ]:
pl.read_csv(QUALITY / "metricas_silver.csv").filter(pl.col("fuente") == "agricola")